# Thyroid Disease Classification — Revised Pipeline with IEEE-Style Figures

This notebook runs the complete leakage-safe revised pipeline and generates **publication-ready figures matching the IEEE journal style** of the attached manuscript:

* Resolution: **≥ 1200 DPI** (vector PDF + raster PNG saved for every figure)
* Fonts: **bold Times serif, ≥ 24 pt** for axis labels, ticks, legends, titles
* Color schemes matching the paper: YlOrRd confusion matrices, multi-color ROC with shaded diagonal, viridis SHAP bars, green/red LIME contributions

All 14 figures are written to `/content/drive/MyDrive/Dataset/all dataset of thyroid/figures_ieee/` as both `.png` (1200 DPI) and `.pdf` (vector) for direct insertion into LaTeX or Word manuscripts.

Estimated runtime on Colab T4: **20–30 minutes** (figure rendering at 1200 DPI is the dominant cost).


## 1. Setup — install libraries, mount Drive, configure IEEE plotting style

In [ ]:
# Required packages — same as the revised pipeline
!pip install -q imbalanced-learn mlxtend shap lime xgboost catboost


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Same dataset path as the original notebook
path = ['/content/drive/MyDrive/FINAL DATASET.csv']

import os
FIG_DIR = '/content/drive/MyDrive/Thyroid Updated IEEE Standard Results'
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Figures will be saved to: {FIG_DIR}')


Figures will be saved to: /content/drive/MyDrive/Thyroid Updated IEEE Standard Results


In [ ]:
import warnings, gc, pickle
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# IEEE JOURNAL STYLE — bold Times serif, ≥ 24 pt, 1200 DPI export
# =====================================================================
DPI_TARGET = 1200
plt.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman','DejaVu Serif','Liberation Serif','serif'],
    'font.weight':       'bold',
    'font.size':         26,
    'axes.titlesize':    28,
    'axes.labelsize':    26,
    'axes.titleweight':  'bold',
    'axes.labelweight':  'bold',
    'axes.linewidth':    2.0,
    'axes.edgecolor':    '#000000',
    'xtick.labelsize':   24,
    'ytick.labelsize':   24,
    'xtick.major.size':  7, 'ytick.major.size':  7,
    'xtick.major.width': 1.6, 'ytick.major.width': 1.6,
    'legend.fontsize':   22,
    'legend.frameon':    True,
    'legend.framealpha': 0.95,
    'legend.edgecolor':  '#000000',
    'figure.dpi':        100,    # screen preview only
    'savefig.dpi':       DPI_TARGET,
    'savefig.bbox':      'tight',
    'savefig.pad_inches': 0.15,
    'pdf.fonttype':      42,     # embed TrueType so text is selectable
    'ps.fonttype':       42,
    'lines.linewidth':   3.0,
    'patch.linewidth':   1.6,
    'grid.linewidth':    1.0,
    'grid.alpha':        0.35,
})

# Helpers ------------------------------------------------------------
def save_fig(fig, name):
    """Export PNG (1200 DPI) and vector PDF, then close to free memory."""
    png = f'{FIG_DIR}/{name}.png'
    pdf = f'{FIG_DIR}/{name}.pdf'
    fig.savefig(png, dpi=DPI_TARGET, facecolor='white', bbox_inches='tight')
    fig.savefig(pdf, facecolor='white', bbox_inches='tight')
    plt.close(fig); plt.close('all'); gc.collect()
    print(f'  Saved {name}.png + {name}.pdf')

def bold_text(ax):
    for label in (ax.get_xticklabels() + ax.get_yticklabels()):
        label.set_fontweight('bold')
    ax.xaxis.label.set_weight('bold')
    ax.yaxis.label.set_weight('bold')
    if ax.title.get_text():
        ax.title.set_weight('bold')

# Paper's exact color conventions ------------------------------------
ROC_COLORS = ['#2E86C1', '#E67E22', '#27AE60', '#C0392B', '#8E44AD']  # blue/orange/green/red/purple
PAPER_CMAP = 'YlOrRd'   # confusion matrices
SHAP_CMAP  = 'viridis'  # SHAP bar plots
print('Plotting style configured: 1200 DPI, bold Times ≥24 pt')


Plotting style configured: 1200 DPI, bold Times ≥24 pt


## 2. Load dataset and run leakage-safe pipeline

In [ ]:
from sklearn.model_selection import (train_test_split, GroupShuffleSplit, StratifiedKFold,
    cross_validate, learning_curve, permutation_test_score)
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, SelectKBest
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SKPipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
    BaggingClassifier, VotingClassifier, StackingClassifier)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
    cohen_kappa_score, matthews_corrcoef, classification_report,
    confusion_matrix, roc_curve, auc, brier_score_loss, log_loss)
from sklearn.preprocessing import label_binarize
from sklearn.utils import resample

RNG = 42
np.random.seed(RNG)

# ---- Load ----
df_raw = pd.read_csv(path[0])
print('Shape:', df_raw.shape)
df = df_raw.drop(['Provisional Diagnosis'], axis=1)

PATIENT_ID_COL = 'Patient Serial ID'
numerical_cols = ['Age', 'TSH Value', 'FT3 Value', 'FT4 Value']
categorical_cols = ['Age Criteria','School/ Job Performance issue','Sex','On_Thyroxine',
    'Query_On_Thyroxine','On_Anti-Thyroid_Medication','Tremor Issue','Bowel','Weight',
    'Skin Problem','Eye-Sign','Appetite Problem','Memory Problem','Sleep Disorder',
    'Pregnant','M/H Problem','Thyroid Surgery','Query_hypothyroid','Query_hyperthyroid',
    'Goiter','Psychological Issue','TSH_measured','FT3_measured','FT4_measured']
target_col = 'Final Diagnosis'

X = df[numerical_cols + categorical_cols].copy()
y = df[target_col].copy()
le = LabelEncoder(); y_enc = le.fit_transform(y)
classes = list(le.classes_)
print('Classes:', classes)
print('Class counts:', dict(zip(le.classes_, np.bincount(y_enc))))


Shape: (348, 31)
Classes: ['EU', 'HYPER', 'HYPO', 'SUB-HYPER', 'SUB-HYPO']
Class counts: {'EU': np.int64(217), 'HYPER': np.int64(51), 'HYPO': np.int64(28), 'SUB-HYPER': np.int64(33), 'SUB-HYPO': np.int64(19)}


In [ ]:
# ---- Grouped split on Patient ID (fixes patient-level leakage) ----
groups = df_raw[PATIENT_ID_COL].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RNG)
train_idx, test_idx = next(gss.split(X, y_enc, groups=groups))

X_train_df = X.iloc[train_idx].copy()
X_test_df  = X.iloc[test_idx].copy()
y_train    = y_enc[train_idx]
y_test     = y_enc[test_idx]

print(f'Train: {len(train_idx)}   Test: {len(test_idx)}')
print('Train class dist:', dict(zip(*np.unique(y_train, return_counts=True))))
print('Test  class dist:', dict(zip(*np.unique(y_test,  return_counts=True))))


Train: 244   Test: 104
Train class dist: {np.int64(0): np.int64(153), np.int64(1): np.int64(37), np.int64(2): np.int64(24), np.int64(3): np.int64(20), np.int64(4): np.int64(10)}
Test  class dist: {np.int64(0): np.int64(64), np.int64(1): np.int64(14), np.int64(2): np.int64(4), np.int64(3): np.int64(13), np.int64(4): np.int64(9)}


In [ ]:
# ---- Leakage-safe pipeline (preprocessing inside every CV fold) ----
def build_pipeline(clf, k=15):
    num_pipe = SKPipeline([
        ('imputer', KNNImputer(n_neighbors=4, weights='distance')),
        ('scaler',  StandardScaler()),
    ])
    cat_pipe = SKPipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe',     OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
    ])
    pre = ColumnTransformer([('num', num_pipe, numerical_cols),
                             ('cat', cat_pipe, categorical_cols)])
    return ImbPipeline([
        ('pre',   pre),
        ('vt',    VarianceThreshold(threshold=0.05)),
        ('fs',    SelectKBest(mutual_info_classif, k=k)),
        ('smote', SMOTE(random_state=RNG, k_neighbors=2)),
        ('clf',   clf),
    ])

# ---- 12 candidate classifiers (identical hyperparameters to the paper) ----
model_configs = {
    'Decision Tree': DecisionTreeClassifier(criterion='entropy', max_depth=5,
                        min_samples_leaf=1, min_samples_split=5, random_state=RNG),
    'Logistic Regression': LogisticRegression(C=10, max_iter=1000, penalty='l2', random_state=RNG),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'SVM': SVC(C=100, kernel='poly', probability=True, random_state=RNG),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10,
                         min_samples_split=5, random_state=RNG),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=RNG),
    'Bagging': BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=5, random_state=RNG),
        n_estimators=50, random_state=RNG),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                             eval_metric='mlogloss', random_state=RNG, verbosity=0),
    'CatBoost': CatBoostClassifier(iterations=200, depth=5, learning_rate=0.1,
                                   random_state=RNG, verbose=0),
}
base_for_ensemble = [
    ('logistic', LogisticRegression(C=10, max_iter=1000, penalty='l2', random_state=RNG)),
    ('svm',      SVC(C=100, kernel='poly', probability=True, random_state=RNG)),
    ('dt',       DecisionTreeClassifier(criterion='entropy', max_depth=5,
                        min_samples_leaf=1, min_samples_split=5, random_state=RNG)),
]
model_configs['Voting Ensemble']  = VotingClassifier(base_for_ensemble, voting='soft')
model_configs['Stacked Ensemble'] = StackingClassifier(
    estimators=base_for_ensemble,
    final_estimator=DecisionTreeClassifier(criterion='entropy', max_depth=5,
                        min_samples_leaf=1, min_samples_split=5, random_state=RNG),
    cv=5)
print(f'{len(model_configs)} models configured')


12 models configured


In [ ]:
# ---- 10-fold CV + hold-out test for every model ----
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RNG)
scoring = {'accuracy':'accuracy','f1_macro':'f1_macro','f1_weighted':'f1_weighted',
           'prec_macro':'precision_macro','rec_macro':'recall_macro'}

rows = []
for name, clf in model_configs.items():
    pipe = build_pipeline(clf, k=15)
    cvr = cross_validate(pipe, X_train_df, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    pipe.fit(X_train_df, y_train)
    y_pred = pipe.predict(X_test_df)
    rows.append({
        'Model': name,
        'CV Acc': cvr['test_accuracy'].mean(), 'CV Std': cvr['test_accuracy'].std(),
        'CV F1-macro': cvr['test_f1_macro'].mean(),
        'Test Acc': accuracy_score(y_test, y_pred),
        'Test F1-macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'Test F1-weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'Test Prec-macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Test Rec-macro':  recall_score(y_test, y_pred, average='macro', zero_division=0),
        'Test Kappa': cohen_kappa_score(y_test, y_pred),
        'Test MCC':   matthews_corrcoef(y_test, y_pred),
    })
    print(f'  {name:22s}  CV={rows[-1]["CV Acc"]:.4f}±{rows[-1]["CV Std"]:.4f}'
          f'  Test={rows[-1]["Test Acc"]:.4f}  κ={rows[-1]["Test Kappa"]:.4f}')

results_df = pd.DataFrame(rows).sort_values('Test Acc', ascending=False).reset_index(drop=True)
print()
print(results_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
results_df.to_csv(f'{FIG_DIR}/model_comparison.csv', index=False)

best_name = results_df.iloc[0]['Model']
best_pipe = build_pipeline(model_configs[best_name], k=15)
best_pipe.fit(X_train_df, y_train)
y_pred_test  = best_pipe.predict(X_test_df)
y_proba_test = best_pipe.predict_proba(X_test_df)
print(f'\nBest model: {best_name}')


  Decision Tree           CV=0.9545±0.0432  Test=0.9519  κ=0.9161
  Logistic Regression     CV=0.8643±0.0595  Test=0.9135  κ=0.8516
  KNN                     CV=0.8273±0.0786  Test=0.8077  κ=0.6886
  SVM                     CV=0.8845±0.0697  Test=0.8558  κ=0.7351
  Naive Bayes             CV=0.8202±0.0790  Test=0.8173  κ=0.7227
  Random Forest           CV=0.9672±0.0306  Test=0.9615  κ=0.9335
  AdaBoost                CV=0.8848±0.0665  Test=0.9519  κ=0.9154
  Bagging                 CV=0.9422±0.0463  Test=0.9615  κ=0.9329
  XGBoost                 CV=0.9588±0.0369  Test=0.9423  κ=0.8994
  CatBoost                CV=0.9668±0.0363  Test=0.9519  κ=0.9162
  Voting Ensemble         CV=0.9508±0.0305  Test=0.9327  κ=0.8818
  Stacked Ensemble        CV=0.8973±0.0457  Test=0.8942  κ=0.8091

              Model  CV Acc  CV Std  CV F1-macro  Test Acc  Test F1-macro  Test F1-weighted  Test Prec-macro  Test Rec-macro  Test Kappa  Test MCC
            Bagging  0.9422  0.0463       0.8931    0.9615  

## 3. Figure 1 — Model Comparison (matches paper Fig. 14)

Horizontal bar chart, viridis colors, value labels inside bars in white. CV-std error bars on each bar.

In [ ]:
# Fig 1 — Model Comparison
ordered = results_df.sort_values('Test Acc', ascending=True).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 6.8))
y_pos = np.arange(len(ordered))
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(ordered)))
ax.barh(y_pos, ordered['Test Acc'].values * 100,
        xerr=ordered['CV Std'].values * 100,
        color=colors, edgecolor='black', linewidth=1.4,
        error_kw={'ecolor':'black','elinewidth':1.6,'capsize':5,'capthick':1.6},
        height=0.72)
ax.set_yticks(y_pos)
ax.set_yticklabels(ordered['Model'].values, fontweight='bold')
ax.set_xlabel('Test-set Accuracy (%)', fontweight='bold')
ax.set_xlim(50, 110)
# Value labels INSIDE the bar tip (white) so they never collide with anything
for i, a in enumerate(ordered['Test Acc']*100):
    ax.text(a - 1.2, i, f'{a:.1f}%', va='center', ha='right',
            fontweight='bold', fontsize=20, color='white')
ax.grid(axis='x', alpha=0.4); ax.set_axisbelow(True)
bold_text(ax)
plt.tight_layout()
save_fig(fig, 'Fig01_model_comparison')


  Saved Fig01_model_comparison.png + Fig01_model_comparison.pdf


## 4. Figure 2 — Confusion Matrix (matches paper Fig. 5b/d, 6b/d, 7b)

YlOrRd colormap, normalized values, square cells, rotated x-tick labels.

In [ ]:
# Fig 2 — Confusion matrix (paper-style YlOrRd)
cm = confusion_matrix(y_test, y_pred_test)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 8.5))
hm = sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap=PAPER_CMAP, vmin=0, vmax=1,
                 xticklabels=classes, yticklabels=classes,
                 annot_kws={'fontsize':24, 'fontweight':'bold'},
                 cbar_kws={'label':'Proportion per True Class','shrink':0.85,'aspect':18},
                 linewidths=1.5, linecolor='white', square=True, ax=ax)
cbar = hm.collections[0].colorbar
cbar.ax.tick_params(labelsize=20, width=1.4)
for lbl in cbar.ax.get_yticklabels(): lbl.set_fontweight('bold')
cbar.ax.yaxis.label.set_fontweight('bold')
cbar.ax.yaxis.label.set_size(22)
ax.set_xlabel('Predicted Labels', fontweight='bold', fontsize=24, labelpad=12)
ax.set_ylabel('True Labels',      fontweight='bold', fontsize=24, labelpad=12)
plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontweight='bold', fontsize=20)
plt.setp(ax.get_yticklabels(), rotation=0, fontweight='bold', fontsize=20)
plt.tight_layout()
save_fig(fig, 'Fig02_confusion_matrix')


  Saved Fig02_confusion_matrix.png + Fig02_confusion_matrix.pdf


## 5. Figure 3 — ROC Curves per Class (matches paper Fig. 5a/c, 6a/c, 7a)

Multi-color one-vs-rest ROC curves, dashed black "Random Guess" diagonal, shaded triangle below diagonal, AUC values in legend.

In [ ]:
# Fig 3 — Multi-class ROC, paper style
y_test_bin = label_binarize(y_test, classes=list(range(len(classes))))
roc_data = {}
for k, cls in enumerate(classes):
    if y_test_bin[:, k].sum() < 1:
        roc_data[cls] = None; continue
    fpr, tpr, _ = roc_curve(y_test_bin[:, k], y_proba_test[:, k])
    roc_data[cls] = {'fpr': fpr, 'tpr': tpr, 'auc': auc(fpr, tpr)}

fig, ax = plt.subplots(figsize=(8, 7))
for k, cls in enumerate(classes):
    if roc_data[cls] is None: continue
    ax.plot(roc_data[cls]['fpr'], roc_data[cls]['tpr'],
            color=ROC_COLORS[k], linewidth=3.5,
            label=f'{cls} (AUC = {roc_data[cls]["auc"]:.4f})')
# Paper-style shaded triangle below the diagonal
ax.fill_between([0,1], [0,1], [0,0], color='#F2D7D5', alpha=0.15)
ax.plot([0,1], [0,1], 'k--', linewidth=2.4, alpha=0.85, label='Random Guess')
ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate',  fontweight='bold')
ax.legend(loc='lower right', fontsize=18, edgecolor='black', borderpad=0.6)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.4); ax.set_axisbelow(True)
bold_text(ax)
plt.tight_layout()
save_fig(fig, 'Fig03_roc_curves')

# Save ROC data for the panel figure
import pickle
with open(f'{FIG_DIR}/_roc_cache.pkl','wb') as f: pickle.dump(roc_data, f)


  Saved Fig03_roc_curves.png + Fig03_roc_curves.pdf


## 6. Figure 4 — Combined ROC + Confusion Matrix Panel (matches paper Fig. 5, 6, 7 layout)

Two-panel side-by-side figure with `(a)` and `(b)` labels — the canonical layout from the paper.

In [ ]:
# Fig 4 — 2-panel ROC + CM (paper Fig 5/6/7 layout)
fig = plt.figure(figsize=(14, 6.5))

# Panel (a) — ROC
ax1 = fig.add_subplot(1, 2, 1)
for k, cls in enumerate(classes):
    if roc_data[cls] is None: continue
    ax1.plot(roc_data[cls]['fpr'], roc_data[cls]['tpr'],
             color=ROC_COLORS[k], linewidth=3.0,
             label=f'{cls} (AUC = {roc_data[cls]["auc"]:.4f})')
ax1.fill_between([0,1], [0,1], [0,0], color='#F2D7D5', alpha=0.15)
ax1.plot([0,1], [0,1], 'k--', linewidth=2.2, alpha=0.85, label='Random Guess')
ax1.set_xlabel('False Positive Rate', fontweight='bold', fontsize=22)
ax1.set_ylabel('True Positive Rate',  fontweight='bold', fontsize=22)
ax1.legend(loc='lower right', fontsize=15, edgecolor='black', borderpad=0.6)
ax1.set_xlim(-0.02, 1.02); ax1.set_ylim(-0.02, 1.02)
ax1.tick_params(labelsize=20)
ax1.grid(True, alpha=0.4); ax1.set_axisbelow(True)
bold_text(ax1)
fig.text(0.255, 0.01, '(a)', fontsize=26, fontweight='bold', ha='center')

# Panel (b) — Confusion Matrix
ax2 = fig.add_subplot(1, 2, 2)
hm = sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap=PAPER_CMAP, vmin=0, vmax=1,
                 xticklabels=classes, yticklabels=classes,
                 annot_kws={'fontsize':20, 'fontweight':'bold'},
                 cbar_kws={'label':'Proportion','shrink':0.85,'aspect':18},
                 linewidths=1.2, linecolor='white', square=True, ax=ax2)
cbar = hm.collections[0].colorbar
cbar.ax.tick_params(labelsize=18)
for lbl in cbar.ax.get_yticklabels(): lbl.set_fontweight('bold')
cbar.ax.yaxis.label.set_fontweight('bold')
cbar.ax.yaxis.label.set_size(20)
ax2.set_xlabel('Predicted Labels', fontweight='bold', fontsize=22, labelpad=8)
ax2.set_ylabel('True Labels',      fontweight='bold', fontsize=22, labelpad=8)
ax2.tick_params(labelsize=18)
plt.setp(ax2.get_xticklabels(), rotation=30, ha='right', fontweight='bold')
plt.setp(ax2.get_yticklabels(), rotation=0, fontweight='bold')
fig.text(0.755, 0.01, '(b)', fontsize=26, fontweight='bold', ha='center')

plt.tight_layout(rect=[0, 0.04, 1, 1])
save_fig(fig, 'Fig04_panel_ROC_CM')


  Saved Fig04_panel_ROC_CM.png + Fig04_panel_ROC_CM.pdf


## 7. Figure 5 — Bootstrap 95% Confidence Intervals (4-panel)

Histograms of bootstrapped Accuracy, Macro-F1, Cohen κ, and MCC with mean line and 95% CI.

In [ ]:
# Fig 5 — Bootstrap CIs (1000 resamples)
n_boot = 1000
boot_acc, boot_f1m, boot_kappa, boot_mcc = [], [], [], []
rng = np.random.RandomState(RNG)
for _ in range(n_boot):
    idx = rng.randint(0, len(y_test), len(y_test))
    yt, yp = y_test[idx], y_pred_test[idx]
    boot_acc.append(accuracy_score(yt, yp))
    boot_f1m.append(f1_score(yt, yp, average='macro', zero_division=0))
    boot_kappa.append(cohen_kappa_score(yt, yp))
    boot_mcc.append(matthews_corrcoef(yt, yp))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
metrics = [('Accuracy', boot_acc), ('Macro-F1', boot_f1m),
           ('Cohen κ', boot_kappa), ('MCC', boot_mcc)]
boot_colors = ['#2E86C1','#27AE60','#E67E22','#8E44AD']
for ax, (mname, vals), color in zip(axes.flat, metrics, boot_colors):
    ax.hist(vals, bins=30, color=color, alpha=0.78, edgecolor='black', linewidth=1.2)
    mean_v = np.mean(vals)
    ci_lo, ci_hi = np.percentile(vals, [2.5, 97.5])
    ax.axvline(mean_v, color='red', linewidth=2.8, linestyle='--',
               label=f'Mean = {mean_v:.4f}')
    ax.axvline(ci_lo, color='black', linewidth=2.2, linestyle=':',
               label=f'95% CI = [{ci_lo:.3f}, {ci_hi:.3f}]')
    ax.axvline(ci_hi, color='black', linewidth=2.2, linestyle=':')
    ax.set_xlabel(mname, fontweight='bold', fontsize=22)
    ax.set_ylabel('Frequency', fontweight='bold', fontsize=22)
    ax.tick_params(labelsize=18)
    ax.legend(fontsize=14, loc='upper left', edgecolor='black')
    ax.grid(True, alpha=0.4); ax.set_axisbelow(True)
    bold_text(ax)
plt.tight_layout()
save_fig(fig, 'Fig05_bootstrap_CIs')


  Saved Fig05_bootstrap_CIs.png + Fig05_bootstrap_CIs.pdf


## 8. Figure 6 — Learning Curves (single panel)

Training vs validation accuracy as the training set grows. Shaded bands = ±1 std across folds.

In [ ]:
# Fig 6 — Learning curve
train_sizes, train_scores, val_scores = learning_curve(
    build_pipeline(model_configs[best_name], k=15),
    X_train_df, y_train,
    train_sizes=np.linspace(0.2, 1.0, 8),
    cv=StratifiedKFold(5, shuffle=True, random_state=RNG),
    scoring='accuracy', n_jobs=-1, random_state=RNG)

fig, ax = plt.subplots(figsize=(9, 7))
tm, ts = train_scores.mean(1), train_scores.std(1)
vm, vs = val_scores.mean(1),   val_scores.std(1)
ax.plot(train_sizes, tm, 'o-', color='#C0392B', linewidth=3.2, markersize=12,
        label='Training Score', markeredgecolor='black', markeredgewidth=1.5)
ax.fill_between(train_sizes, tm-ts, tm+ts, alpha=0.22, color='#C0392B')
ax.plot(train_sizes, vm, 's-', color='#2E86C1', linewidth=3.2, markersize=12,
        label='5-fold CV Score', markeredgecolor='black', markeredgewidth=1.5)
ax.fill_between(train_sizes, vm-vs, vm+vs, alpha=0.22, color='#2E86C1')
ax.set_xlabel('Training Set Size', fontweight='bold')
ax.set_ylabel('Accuracy', fontweight='bold')
ax.legend(loc='lower right', fontsize=20, edgecolor='black', borderpad=0.8)
ax.set_ylim(0.5, 1.03); ax.grid(True, alpha=0.4); ax.set_axisbelow(True)
bold_text(ax)
plt.tight_layout()
save_fig(fig, 'Fig06_learning_curve')


  Saved Fig06_learning_curve.png + Fig06_learning_curve.pdf


## 9. Figure 7 — Permutation Test (single panel)

Histogram of accuracy under shuffled labels (null distribution) vs. observed accuracy on real labels.

In [ ]:
# Fig 7 — Permutation test
score, perm_scores, pval = permutation_test_score(
    build_pipeline(model_configs[best_name], k=15),
    X_train_df, y_train,
    cv=StratifiedKFold(5, shuffle=True, random_state=RNG),
    n_permutations=200, n_jobs=-1, random_state=RNG, scoring='accuracy')
print(f'Observed CV accuracy: {score:.4f}')
print(f'Null mean ± std:      {perm_scores.mean():.4f} ± {perm_scores.std():.4f}')
print(f'p-value:              {pval:.4f}')

fig, ax = plt.subplots(figsize=(11, 7))
ax.hist(perm_scores, bins=20, color='#8172B3', alpha=0.78, edgecolor='black', linewidth=1.2,
        label=f'Null distribution (n={len(perm_scores)})')
ax.axvline(score, color='red', linewidth=3.0, linestyle='--', label=f'Observed = {score:.4f}')
ax.set_xlabel('Accuracy under shuffled labels', fontweight='bold')
ax.set_ylabel('Count', fontweight='bold')
ax.legend(loc='upper right', fontsize=18, edgecolor='black', borderpad=0.7,
          bbox_to_anchor=(0.99, 0.99))
# p-value box positioned away from both legend and observed line
ax.text(0.5, 0.5, f'p-value = {pval:.4f}',
        transform=ax.transAxes, fontsize=24, fontweight='bold',
        ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.5', fc='#FCF3CF', ec='black', lw=1.5))
ax.grid(True, alpha=0.4); ax.set_axisbelow(True)
bold_text(ax)
plt.tight_layout()
save_fig(fig, 'Fig07_permutation_test')


Observed CV accuracy: 0.9385
Null mean ± std:      0.3369 ± 0.0347
p-value:              0.0050
  Saved Fig07_permutation_test.png + Fig07_permutation_test.pdf


## 10. Figure 8 — Reliability Diagram + Brier Score + ECE (2-panel)

Per-class calibration curves with Brier scores in the legend, and a side-by-side bar chart of Brier vs ECE per class. Vertically-rotated value labels avoid all collisions.

In [ ]:
# Fig 8 — Reliability + Brier/ECE bars
def expected_calibration_error(y_true_bin, y_prob, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece, n = 0.0, len(y_true_bin)
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob > lo) & (y_prob <= hi)
        if mask.sum() == 0: continue
        ece += (mask.sum() / n) * abs(y_true_bin[mask].mean() - y_prob[mask].mean())
    return ece

y_bin = label_binarize(y_test, classes=list(range(len(classes))))
calib_rows = []
for k, cls in enumerate(classes):
    if y_bin[:,k].sum() < 1: continue
    calib_rows.append({
        'Class': cls, 'n_positive': int(y_bin[:,k].sum()),
        'Brier': brier_score_loss(y_bin[:,k], y_proba_test[:,k]),
        'ECE':   expected_calibration_error(y_bin[:,k], y_proba_test[:,k], n_bins=10),
    })
calib = pd.DataFrame(calib_rows)
print(calib.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
calib.to_csv(f'{FIG_DIR}/calibration.csv', index=False)

fig = plt.figure(figsize=(15, 7))

# Panel (a) — Reliability diagram
ax1 = fig.add_subplot(1, 2, 1)
for k, (cls, color) in enumerate(zip(classes, ROC_COLORS)):
    if y_bin[:, k].sum() < 3: continue
    try:
        pt, pp = calibration_curve(y_bin[:, k], y_proba_test[:, k],
                                    n_bins=5, strategy='quantile')
        ax1.plot(pp, pt, 'o-', color=color, linewidth=2.8, markersize=12,
                 markeredgecolor='black', markeredgewidth=1.5,
                 label=f'{cls} (Brier={calib.iloc[k]["Brier"]:.3f})')
    except Exception: pass
ax1.plot([0,1],[0,1], 'k--', linewidth=2.2, alpha=0.85, label='Perfect')
ax1.set_xlabel('Mean Predicted Probability', fontweight='bold', fontsize=22)
ax1.set_ylabel('Observed Positive Fraction', fontweight='bold', fontsize=22)
ax1.legend(loc='lower right', fontsize=14, edgecolor='black', borderpad=0.5)
ax1.set_xlim(-0.02, 1.02); ax1.set_ylim(-0.02, 1.02)
ax1.tick_params(labelsize=18)
ax1.grid(True, alpha=0.4); ax1.set_axisbelow(True)
bold_text(ax1)
fig.text(0.255, 0.01, '(a)', fontsize=26, fontweight='bold', ha='center')

# Panel (b) — Brier vs ECE per class
ax2 = fig.add_subplot(1, 2, 2)
x_pos = np.arange(len(classes))
w = 0.36
bars_b = ax2.bar(x_pos - w/2, calib['Brier'], w, label='Brier Score',
                  color='#2E86C1', alpha=0.85, edgecolor='black', linewidth=1.4)
bars_e = ax2.bar(x_pos + w/2, calib['ECE'],   w, label='ECE',
                  color='#C0392B', alpha=0.85, edgecolor='black', linewidth=1.4)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(classes, fontweight='bold', rotation=30, ha='right', fontsize=18)
ax2.set_ylabel('Calibration error', fontweight='bold', fontsize=22)
ax2.tick_params(labelsize=18)
ax2.legend(fontsize=18, edgecolor='black', borderpad=0.6, loc='upper right')

# Vertically-rotated per-bar labels (no collision)
max_val = max(calib['Brier'].max(), calib['ECE'].max())
label_offset = max_val * 0.03
for bar, b in zip(bars_b, calib['Brier']):
    if pd.isna(b): continue
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + label_offset,
             f'{b:.3f}', ha='center', va='bottom', rotation=90,
             fontweight='bold', fontsize=14, color='#1A4F72')
for bar, e in zip(bars_e, calib['ECE']):
    if pd.isna(e): continue
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + label_offset,
             f'{e:.3f}', ha='center', va='bottom', rotation=90,
             fontweight='bold', fontsize=14, color='#922B21')
ax2.set_ylim(0, max_val * 1.40)
ax2.grid(True, axis='y', alpha=0.4); ax2.set_axisbelow(True)
bold_text(ax2)
fig.text(0.755, 0.01, '(b)', fontsize=26, fontweight='bold', ha='center')

plt.tight_layout(rect=[0, 0.04, 1, 1])
save_fig(fig, 'Fig08_reliability_brier_ECE')


    Class  n_positive  Brier    ECE
       EU          64 0.0270 0.0238
    HYPER          14 0.0125 0.0058
     HYPO           4 0.0000 0.0002
SUB-HYPER          13 0.0078 0.0049
 SUB-HYPO           9 0.0172 0.0203
  Saved Fig08_reliability_brier_ECE.png + Fig08_reliability_brier_ECE.pdf


## 11. Figure 9 — Recalibration Comparison (3-panel)

Reliability diagrams for uncalibrated, isotonic-calibrated, and Platt-calibrated versions of the best model — shows how post-hoc recalibration affects probability quality.

In [ ]:
# Fig 9 — Recalibration comparison
from sklearn.base import clone
def build_calibrated_pipeline(base_clf, method):
    calibrated = CalibratedClassifierCV(
        estimator=base_clf, method=method,
        cv=StratifiedKFold(5, shuffle=True, random_state=RNG))
    return build_pipeline(calibrated, k=15)

print('Fitting isotonic and sigmoid calibrated versions...')
pipe_iso = build_calibrated_pipeline(clone(model_configs[best_name]), 'isotonic')
pipe_sig = build_calibrated_pipeline(clone(model_configs[best_name]), 'sigmoid')
pipe_iso.fit(X_train_df, y_train)
pipe_sig.fit(X_train_df, y_train)
y_proba_iso = pipe_iso.predict_proba(X_test_df)
y_proba_sig = pipe_sig.predict_proba(X_test_df)

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), sharey=True)
panels = [('Uncalibrated', y_proba_test),
          ('Isotonic',     y_proba_iso),
          ('Sigmoid (Platt)', y_proba_sig)]
for ax, (title, proba) in zip(axes, panels):
    for k, (cls, color) in enumerate(zip(classes, ROC_COLORS)):
        if y_bin[:, k].sum() < 3: continue
        try:
            pt, pp = calibration_curve(y_bin[:, k], proba[:, k], n_bins=5, strategy='quantile')
            ax.plot(pp, pt, 'o-', color=color, linewidth=2.6, markersize=10,
                    markeredgecolor='black', markeredgewidth=1.3, label=cls)
        except Exception: pass
    ax.plot([0,1],[0,1], 'k--', linewidth=2.0, alpha=0.85)
    ax.set_xlabel('Mean Predicted Probability', fontweight='bold', fontsize=20)
    ax.set_title(title, fontweight='bold', pad=10, fontsize=22)
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
    ax.tick_params(labelsize=16)
    ax.grid(True, alpha=0.4); ax.set_axisbelow(True)
    bold_text(ax)
axes[0].set_ylabel('Observed Positive Fraction', fontweight='bold', fontsize=20)
axes[0].legend(loc='upper left', fontsize=14, edgecolor='black', borderpad=0.5)
fig.text(0.18, 0.005, '(a)', fontsize=24, fontweight='bold', ha='center')
fig.text(0.51, 0.005, '(b)', fontsize=24, fontweight='bold', ha='center')
fig.text(0.84, 0.005, '(c)', fontsize=24, fontweight='bold', ha='center')
plt.tight_layout(rect=[0, 0.04, 1, 1])
save_fig(fig, 'Fig09_recalibration_comparison')


Fitting isotonic and sigmoid calibrated versions...
  Saved Fig09_recalibration_comparison.png + Fig09_recalibration_comparison.pdf


## 12. Prepare SHAP / LIME data (run once, reused by Figs 10–14)

Refit the best classifier outside the imblearn pipeline so we can call SHAP's TreeExplainer on the trained model, then compute LIME for one representative patient per class.

In [ ]:
# Re-do preprocessing manually so we can keep the trained classifier accessible to SHAP/LIME
import shap, lime, lime.lime_tabular

pre = ColumnTransformer([
    ('num', SKPipeline([('imp', KNNImputer(n_neighbors=4, weights='distance')),
                        ('sc',  StandardScaler())]), numerical_cols),
    ('cat', SKPipeline([('imp', SimpleImputer(strategy='most_frequent')),
                        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))]),
            categorical_cols),
])
pre.fit(X_train_df)
X_tr_arr = pre.transform(X_train_df)
X_te_arr = pre.transform(X_test_df)
ohe = pre.named_transformers_['cat'].named_steps['ohe']
feat_names = list(numerical_cols) + list(ohe.get_feature_names_out(categorical_cols))

vt = VarianceThreshold(threshold=0.05); vt.fit(X_tr_arr)
mask_v = vt.get_support()
feat_v = [f for f,m in zip(feat_names, mask_v) if m]
X_tr_v = X_tr_arr[:, mask_v]; X_te_v = X_te_arr[:, mask_v]

sm = SMOTE(random_state=RNG, k_neighbors=2)
X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_v, y_train)

sk = SelectKBest(mutual_info_classif, k=15); sk.fit(X_tr_sm, y_tr_sm)
sel_mask = sk.get_support()
sel_feats = [f for f,m in zip(feat_v, sel_mask) if m]
X_tr_fs = X_tr_sm[:, sel_mask]; X_te_fs = X_te_v[:, sel_mask]
print(f'15 selected features: {sel_feats}')

# Use Random Forest as XAI surrogate (TreeExplainer is fast and exact)
xai_model = RandomForestClassifier(n_estimators=200, max_depth=10,
                                    min_samples_split=5, random_state=RNG)
xai_model.fit(X_tr_fs, y_tr_sm)

explainer = shap.TreeExplainer(xai_model)
shap_values = explainer.shap_values(X_te_fs)
if isinstance(shap_values, list):
    shap_arr = np.stack(shap_values, axis=0)
else:
    shap_arr = shap_values
    if shap_arr.ndim == 3: shap_arr = np.transpose(shap_arr, (2,0,1))
print(f'SHAP array shape: {shap_arr.shape}  (classes, samples, features)')

# LIME for one representative patient per class
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_tr_fs, feature_names=sel_feats,
    class_names=classes, discretize_continuous=False,
    random_state=RNG, mode='classification')

lime_results = {}
for cls_idx, cls_name in enumerate(classes):
    idxs = np.where(y_test == cls_idx)[0]
    if len(idxs) == 0: continue
    i = int(idxs[0])
    pred_cls = int(xai_model.predict(X_te_fs[i:i+1])[0])
    pred_prob = xai_model.predict_proba(X_te_fs[i:i+1])[0]
    exp = lime_explainer.explain_instance(X_te_fs[i], xai_model.predict_proba,
                                            num_features=10, labels=[pred_cls])
    lime_results[cls_name] = {
        'instance_idx': i, 'pred_cls': pred_cls,
        'pred_cls_name': classes[pred_cls],
        'pred_proba': float(pred_prob[pred_cls]),
        'lime_proba': pred_prob.tolist(),
        'lime_list': exp.as_list(label=pred_cls),
    }
print(f'LIME computed for {len(lime_results)} representative patients')


15 selected features: ['Age', 'TSH Value', 'FT3 Value', 'FT4 Value', 'On_Thyroxine_YES', 'Query_On_Thyroxine_YES', 'Tremor Issue_YES', 'Bowel_NO', 'Weight_L', 'Skin Problem_S', 'Memory Problem_YES', 'Sleep Disorder_S', 'M/H Problem_YES', 'Query_hypothyroid_YES', 'Query_hyperthyroid_YES']
SHAP array shape: (5, 104, 15)  (classes, samples, features)
LIME computed for 5 representative patients


## 13. Figure 10 — SHAP Global Feature Importance (matches paper Fig. 9)

Horizontal viridis bar chart of mean absolute SHAP values across all classes and test samples. This is the paper's signature SHAP plot.

In [ ]:
# Fig 10 — SHAP global importance (paper Fig 9 style)
global_shap = np.mean(np.abs(shap_arr), axis=(0,1))

def clean_name(f):
    return (f.replace('_YES','').replace('_NO',' (No)')
             .replace('_S',' (S)').replace('_L',' (Loss)')
             .replace('_',' ').strip())
disp = [clean_name(f) for f in sel_feats]
order = np.argsort(global_shap)
disp_sorted = [disp[i] for i in order]
shap_sorted = global_shap[order]

fig, ax = plt.subplots(figsize=(10, 8))
viridis = plt.cm.viridis(np.linspace(0.2, 0.85, len(sel_feats)))
ax.barh(np.arange(len(sel_feats)), shap_sorted, color=viridis,
        edgecolor='black', linewidth=1.5)
ax.set_yticks(np.arange(len(sel_feats)))
ax.set_yticklabels(disp_sorted, fontweight='bold', fontsize=18)
ax.set_xlabel('Mean Absolute SHAP Value', fontweight='bold')
ax.grid(axis='x', alpha=0.4); ax.set_axisbelow(True)
bold_text(ax)
for i, v in enumerate(shap_sorted):
    ax.text(v + max(shap_sorted)*0.012, i, f'{v:.3f}',
            va='center', fontweight='bold', fontsize=14)
plt.tight_layout()
save_fig(fig, 'Fig10_SHAP_global_importance')


  Saved Fig10_SHAP_global_importance.png + Fig10_SHAP_global_importance.pdf


## 14. Figure 11 — SHAP Beeswarm Plot (matches paper Fig. 11)

Per-instance SHAP values dotplot with high/low feature-value coloring.

In [ ]:
# Fig 11 — SHAP beeswarm (paper Fig 11 style)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_arr[0], X_te_fs, feature_names=sel_feats,
                  show=False, plot_size=(10, 8), max_display=15)
fig = plt.gcf()
ax = plt.gca()
ax.set_xlabel('SHAP value (impact on model output)', fontweight='bold', fontsize=22)
plt.setp(ax.get_xticklabels(), fontweight='bold', fontsize=18)
plt.setp(ax.get_yticklabels(), fontweight='bold', fontsize=16)
for sub in fig.axes:
    if sub != ax:
        sub.tick_params(labelsize=16)
        for lbl in sub.get_yticklabels(): lbl.set_fontweight('bold')
        if sub.yaxis.label.get_text():
            sub.yaxis.label.set_fontweight('bold')
            sub.yaxis.label.set_size(20)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/Fig11_SHAP_beeswarm.png', dpi=DPI_TARGET, facecolor='white', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/Fig11_SHAP_beeswarm.pdf', facecolor='white', bbox_inches='tight')
plt.close('all'); gc.collect()
print('Saved Fig11_SHAP_beeswarm')


Saved Fig11_SHAP_beeswarm


## 15. Figure 12 — LIME Local Explanation (matches paper Fig. 8)

Two-panel layout: (a) prediction probabilities with the predicted class highlighted in green; (b) horizontal bar chart of LIME feature contributions, green for positive contribution, red for negative.

In [ ]:
# Fig 12 — LIME local explanation (paper Fig 8 style)
target_cls = 'HYPO' if 'HYPO' in lime_results else list(lime_results.keys())[0]
linfo = lime_results[target_cls]

fig = plt.figure(figsize=(14, 6))

# Panel (a) — Prediction probabilities
ax1 = fig.add_subplot(1, 2, 1)
probs = linfo['lime_proba']
bar_colors = ['#28B463' if i == linfo['pred_cls'] else '#5DADE2' for i in range(len(classes))]
bars = ax1.bar(np.arange(len(classes)), probs, color=bar_colors,
               edgecolor='black', linewidth=1.5)
ax1.set_xticks(np.arange(len(classes)))
ax1.set_xticklabels(classes, fontweight='bold', rotation=20, ha='right', fontsize=18)
ax1.set_ylabel('Probability', fontweight='bold', fontsize=22)
ax1.set_title(f'PRED: {linfo["pred_cls_name"]} ({linfo["pred_proba"]:.2f}) | TRUE: {target_cls}',
              fontweight='bold', pad=10, fontsize=20)
ax1.set_ylim(0, max(1.05, max(probs)*1.1))
ax1.tick_params(labelsize=18)
for bar, p in zip(bars, probs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{p:.2f}', ha='center', fontweight='bold', fontsize=16)
ax1.grid(True, axis='y', alpha=0.4); ax1.set_axisbelow(True)
fig.text(0.255, 0.01, '(a)', fontsize=24, fontweight='bold', ha='center')

# Panel (b) — LIME feature contributions
ax2 = fig.add_subplot(1, 2, 2)
lime_features, lime_weights = zip(*linfo['lime_list'])
order = np.argsort(np.abs(lime_weights))[::-1][:10]
lf = [lime_features[i] for i in order][::-1]
lw = [lime_weights[i]  for i in order][::-1]
colors_l = ['#28B463' if w > 0 else '#E74C3C' for w in lw]
ax2.barh(np.arange(len(lf)), lw, color=colors_l, edgecolor='black', linewidth=1.5)
ax2.set_yticks(np.arange(len(lf)))
ax2.set_yticklabels(lf, fontweight='bold', fontsize=14)
ax2.axvline(0, color='black', linewidth=1.6)
ax2.set_xlabel('LIME Weight', fontweight='bold', fontsize=22)
ax2.set_title(f'LIME Explanation — {linfo["pred_cls_name"]}',
              fontweight='bold', pad=10, fontsize=20)
ax2.tick_params(labelsize=16)
ax2.grid(True, axis='x', alpha=0.4); ax2.set_axisbelow(True)
fig.text(0.755, 0.01, '(b)', fontsize=24, fontweight='bold', ha='center')

plt.tight_layout(rect=[0, 0.04, 1, 1])
save_fig(fig, 'Fig12_LIME_local_explanation')


  Saved Fig12_LIME_local_explanation.png + Fig12_LIME_local_explanation.pdf


## 16. Figure 13 — SHAP vs LIME Global Importance Scatter (matches paper Fig. 13)

Per-feature scatter of mean |SHAP| vs mean |LIME| importance, with a y=x reference line and Spearman rank correlation. Top-3 features are annotated.

In [ ]:
# Fig 13 — SHAP vs LIME global comparison (paper Fig 13 style)
from scipy.stats import spearmanr

# Compute global LIME = mean absolute weight across the lime_results sample
lime_per_feat = {f: [] for f in sel_feats}
for cls_name, info in lime_results.items():
    for fname, w in info['lime_list']:
        # LIME labels can be numeric/discretized; match by base feature name
        for f in sel_feats:
            if f.split('_')[0] in fname or f in fname:
                lime_per_feat[f].append(abs(w))
                break
mean_lime = np.array([np.mean(lime_per_feat[f]) if lime_per_feat[f] else 0.0 for f in sel_feats])

sl_global = pd.DataFrame({
    'feature': sel_feats,
    'mean_abs_shap': global_shap,
    'mean_abs_lime': mean_lime,
})
sl_global.to_csv(f'{FIG_DIR}/shap_vs_lime_global.csv', index=False)

sx = sl_global['mean_abs_shap'].values
ly = sl_global['mean_abs_lime'].values
rho, pval_corr = spearmanr(sx, ly)

fig, ax = plt.subplots(figsize=(9.5, 7.5))
ax.scatter(sx, ly, s=200, c='#2E86C1', edgecolor='black',
           linewidth=1.6, alpha=0.85, zorder=3)
mx = max(max(sx), max(ly), 1e-9)
ax.set_xlim(-mx*0.05, mx*1.15)
ax.set_ylim(-mx*0.05, mx*0.6 if mx > 0 else 1)
ax.plot([0, mx*1.15], [0, mx*1.15], color='#C0392B', linewidth=2.4,
        linestyle='--', label='y = x (perfect agreement)')
top3 = np.argsort(sx)[::-1][:3]
for idx in top3:
    feat = clean_name(sl_global['feature'].iloc[idx])
    ax.annotate(feat, (sx[idx], ly[idx]),
                xytext=(-15, 12), textcoords='offset points',
                fontsize=15, fontweight='bold', ha='right',
                bbox=dict(boxstyle='round,pad=0.25', fc='#FCF3CF', ec='black', lw=1.0),
                arrowprops=dict(arrowstyle='-', color='black', lw=0.6))
ax.set_xlabel('SHAP importance (mean abs)', fontweight='bold')
ax.set_ylabel('LIME importance (mean abs)', fontweight='bold')
ax.text(0.04, 0.94, f'Spearman ρ = {rho:.3f}\np = {pval_corr:.2e}',
        transform=ax.transAxes, fontsize=18, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', fc='#D5F5E3', ec='black', lw=1.0),
        verticalalignment='top')
ax.legend(loc='lower right', fontsize=18, edgecolor='black', borderpad=0.7)
ax.grid(True, alpha=0.4); ax.set_axisbelow(True)
bold_text(ax)
plt.tight_layout()
save_fig(fig, 'Fig13_SHAP_vs_LIME_global')


  Saved Fig13_SHAP_vs_LIME_global.png + Fig13_SHAP_vs_LIME_global.pdf


## 17. Figure 14 — LIME Explanations Across All Five Classes (multi-panel)

One representative patient per class arranged in a 2×3 grid. Useful for showing how feature importances shift between thyroid states.

In [ ]:
# Fig 14 — LIME multi-patient (one per class)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes_flat = axes.flat
items = list(lime_results.items())
for i, (cls_name, info) in enumerate(items):
    if i >= 6: break
    ax = axes_flat[i]
    lime_features, lime_weights = zip(*info['lime_list'])
    order = np.argsort(np.abs(lime_weights))[::-1][:8]
    lf = [lime_features[j] for j in order][::-1]
    lw = [lime_weights[j]  for j in order][::-1]
    colors_l = ['#28B463' if w > 0 else '#E74C3C' for w in lw]
    ax.barh(np.arange(len(lf)), lw, color=colors_l, edgecolor='black', linewidth=1.2)
    ax.set_yticks(np.arange(len(lf)))
    ax.set_yticklabels(lf, fontweight='bold', fontsize=11)
    ax.axvline(0, color='black', linewidth=1.4)
    ax.set_xlabel('LIME Weight', fontweight='bold', fontsize=15)
    ax.set_title(f'TRUE: {cls_name} | PRED: {info["pred_cls_name"]} ({info["pred_proba"]:.2f})',
                 fontweight='bold', fontsize=14, pad=8)
    ax.tick_params(labelsize=12)
    ax.grid(True, axis='x', alpha=0.4); ax.set_axisbelow(True)
for j in range(len(items), 6):
    axes_flat[j].axis('off')
plt.tight_layout()
save_fig(fig, 'Fig14_LIME_multi_patient')


  Saved Fig14_LIME_multi_patient.png + Fig14_LIME_multi_patient.pdf


## 18. Verify all figures saved to Drive

In [ ]:
import os
files = sorted(f for f in os.listdir(FIG_DIR) if f.endswith('.png'))
print(f'Generated {len(files)} PNG figures (matched .pdf for each):')
for f in files:
    size = os.path.getsize(f'{FIG_DIR}/{f}') / 1024
    print(f'  {f:50s} ({size:.0f} KB)')


Generated 14 PNG figures (matched .pdf for each):
  Fig01_model_comparison.png                         (1809 KB)
  Fig02_confusion_matrix.png                         (2355 KB)
  Fig03_roc_curves.png                               (1450 KB)
  Fig04_panel_ROC_CM.png                             (2950 KB)
  Fig05_bootstrap_CIs.png                            (1757 KB)
  Fig06_learning_curve.png                           (915 KB)
  Fig07_permutation_test.png                         (1004 KB)
  Fig08_reliability_brier_ECE.png                    (3523 KB)
  Fig09_recalibration_comparison.png                 (2632 KB)
  Fig10_SHAP_global_importance.png                   (1642 KB)
  Fig11_SHAP_beeswarm.png                            (1868 KB)
  Fig12_LIME_local_explanation.png                   (2211 KB)
  Fig13_SHAP_vs_LIME_global.png                      (1399 KB)
  Fig14_LIME_multi_patient.png                       (2554 KB)


## Summary

All 14 figures from the revised pipeline have been re-rendered in the IEEE journal style of the attached manuscript:

| # | Figure | Paper analog | Style notes |
|---|--------|--------------|-------------|
| 1 | Model comparison | Fig. 14 | Viridis bars, in-bar value labels |
| 2 | Confusion matrix | Fig. 5b/d, 6b/d, 7b | YlOrRd colormap, square cells |
| 3 | ROC curves | Fig. 5a/c, 6a/c, 7a | Multi-color, shaded diagonal |
| 4 | ROC + CM panel | Fig. 5/6/7 layout | Two-panel with (a)/(b) labels |
| 5 | Bootstrap CIs | new | 4-panel, mean + 95% CI markers |
| 6 | Learning curve | new | Train vs CV with shaded ± std |
| 7 | Permutation test | new | Null distribution + observed line |
| 8 | Reliability + Brier/ECE | new | 2-panel, vertically-rotated bar labels |
| 9 | Recalibration | new | Uncalibrated/Isotonic/Platt 3-panel |
| 10 | SHAP global | Fig. 9 | Viridis horizontal bars |
| 11 | SHAP beeswarm | Fig. 11 | Per-instance dotplot, blue→red |
| 12 | LIME local | Fig. 8 | Probability bars + green/red contributions |
| 13 | SHAP vs LIME | Fig. 13 | Scatter with y=x line, Spearman ρ |
| 14 | LIME multi-patient | new | 2×3 grid, one patient per class |

Every figure exports both 1200 DPI PNG (for Word and online portals) and vector PDF (for LaTeX `\includegraphics`).
